&nbsp;

# Heat stress-related ATAC-seq Metadata Validation

- ChIP-Atlas のエンリッチメント解析の結果から、熱ストレスなどに関連しているものを探索するため、ここで調査する
- [ChIP-Atlas](https://chip-atlas.org/) に登録されているかどうかを `bool` 値で カラムとして追加する (In progress)
- 機械学習モデルである [ENCODE-rE2G](https://github.com/EngreitzLab/ENCODE_rE2G) を実行することができるため、実行できるように併せて追記もしている

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")

print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Strategy 0. [../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md](../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md)から移行したヒット

- [../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md](../../Data/curation/GEO_other_result/gds_other_result_summary_260123_modified.md)で、ATAC-seq としてヒットしているデータをこちらに移行する

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. GSE105028 (ATAC-seq) (ChIP-Atlas: True)

- ✅ DOWNLOAD (M5 Macbook Pro)
- ✅ trimming using `fastp` (M5 Macbook Pro)
- ✅ `.tagalign` using `Chromap` (M5 Macbook Pro)
- ✅ sort `.tagalign` for `ENCODE-rE2G` (M5 Macbook Pro)
- ✅ Execute `ENCODE-rE2G` (M5 Macbook Pro)

&nbsp;

- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)から更にこちらに移動

&nbsp;

- [Architectural Proteins and Pluripotency Factors Cooperate to Orchestrate the Transcriptional Response of hESCs to Temperature Stress](https://doi.org/10.1016/j.molcel.2018.07.012)
- About `Omni-ATAC-seq`: [An improved ATAC-seq protocol reduces background and enables interrogation of frozen tissues](https://pmc.ncbi.nlm.nih.gov/articles/PMC5623106/)
- 以下のような記述を確認
- `..., we performed ATAC-seq using the FastATAC procedure (Corces et al., 2016) and separated reads in the <115-bp range from those in the 180- to 247-bp range in order to independently map transcription factor binding sites (Tn5 hypersensitive sites, THSSs) and nucleosomes, respectively (Table S1) (Schep et al., 2015).`

- BioProject: [PRJNA1177810](https://www.ncbi.nlm.nih.gov/bioproject/PRJNA1177810)

&nbsp;

### ChIP-Atlas Diff Analysis

- replicate が 2つ以上ある条件を満たさなかったので、`Diff Analysis` には使用しない
- Fast ATAC-seq での組み合わせが可能かと思ったが、`SRX3316067`が ChIP-Atlas に登録されていなかったため、使用しない

In [4]:
GSE105028_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE105028/GSE105028_PEP/GSE105028_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ATAC-seq")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.lit("https://doi.org/10.1016/j.molcel.2018.07.012").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit(False).alias("is_chip_atlas_registered"),
    pl.lit("hESC").alias("cell_line")
)

GSE105028_PEP = GSE105028_PEP[[s.name for s in GSE105028_PEP if not (s.null_count() == GSE105028_PEP.height)]]

GSE105028 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE105028.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE105028_filtered = GSE105028[[s.name for s in GSE105028 if not (s.null_count() == GSE105028.height)]]

GSE105028_combined = GSE105028_PEP.join(
    GSE105028_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

GSE105028_combined = GSE105028_combined[[s.name for s in GSE105028_combined if not (s.null_count() == GSE105028_combined.height)]]

# display(GSE105028_combined)

In [5]:
defined_pairs_gse105028_atacseq = [
    ("SRX3316069", "SRX3316067"), # HS Fast ATAC-Seq Rep1 vs. Fast ATAC-Seq Rep1
    ("SRX3316070", "SRX3316068"), # HS Fast ATAC-Seq Rep2 vs. Fast ATAC-Seq Rep2
    ("SRX3316072", "SRX3316071"), # HS OMNI ATAC-Seq vs. OMNI ATAC-Seq
    ("SRX3316074", "SRX3316073")  # HS BRG1-KO_ATAC-Seq vs. BRG1-KO_ATAC-Seq
]

paired_metadata_df_gse105028_atacseq = create_paired_metadata(
    combined_df=GSE105028_combined,
    defined_sample_pairs=defined_pairs_gse105028_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse105028_atacseq is not None:
    paired_metadata_df_gse105028_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE105028_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse105028_atacseq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,…,experiment_title,sample_accession_right,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX3316069""","""SRX3316067""","""hs_fast_atac-seq_rep1 vs fast_atac-seq_rep1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR6206775 vs SRR6206773""","""HS Fast ATAC-Seq Rep1 vs Fast ATAC-Seq Rep1""","""GSM2829008 vs GSM2829006""","""Public on Aug 03 2018""","""Oct 24 2017""","""May 15 2019""","""SRA""","""1""","""H9 hESCs""","""Homo sapiens""","""9606""","""genomic DNA""","""fast ATAC-Seq, Fast ATAC-Seq under heat shock conditions, HS_ATAC-Seq.bw vs fast ATAC-Seq, Fast ATAC-Seq under normal conditions, ATAC-Seq.bw""","""GPL11154""","""XIAOWEN,,LYU""","""xiaowenlyu@gmail.com""","""+86-18701064502""","""XIAMEN UNIVERSITY""","""No.4221, Xiang'an South Road, Xiang'an District""","""XIAMEN""","""FUJIAN""","""361102""","""China""","""Illumina HiSeq 2000""","""other""","""genomic""","""ATAC-seq""","""GSM2829008 vs GSM2829006""","""NONE""","""GSE105028""",…,"""Illumina HiSeq 2000 sequencing: GSM2829008: HS Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq vs Illumina HiSeq 2000 sequencing: GSM2829006: Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq""","""SAMN07828432 vs SAMN07828392""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""SRA619794""","""other""","""2018-08-04""","""2685800665;2730149674 vs 3903606294;3941685842""","""ILLUMINA""","""Generic""","""HS Fast ATAC-Seq Rep1 vs Fast ATAC-Seq Rep1""","""SRP120027""","""81247114 vs 118170993""","""Architectural proteins and pluripotency factors cooperate to orchestrate the transcriptional response of hESCs to temperature stress""","""GENERATED_FILE;GENERATED_FILE""","""HS Fast ATAC-Seq Rep1 vs Fast ATAC-Seq Rep1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2829008: HS Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq vs Illumina HiSeq 2000 sequencing: GSM2829006: Fast ATAC-Seq Rep1 Homo sapiens ATAC-seq""","""H9 human Embryonic Stem Cell Line""","""8124711400 vs 11817099300""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM2829008 vs GSM2829006""","""GSM2829008_r1 vs GSM2829006_r1""","""For Hi-C, cells were crosslinked with 1% formaldehyde for 10 min at room temperature and quenched with glycine, nuclei were isolated and chromatin digested by HindIII, filled in with biotin-dCTP, and …","""GSM2829008 vs GSM2829006""","""GEO""","""env_tax:terrestrial;env_tax""","""2018-08-04""","""Homo sapiens""","""9606""","""PRJNA41444

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. GSE280307 (ATAC-seq) (ChIP-Atlas: False)

- ✅ DOWNLOAD (M5 Macbook Pro)
- ✅ trimming using `fastp` (M5 Macbook Pro)
- ✅ `.tagalign` using `Chromap` (M5 Macbook Pro)
- ✅ sort `.tagalign` for `ENCODE-rE2G` (M5 Macbook Pro)
<!-- - ✅ Execute `ENCODE-rE2G` (M5 Macbook Pro) -->

&nbsp;

- validate date: 2026-05-09
- [De novo assembly of nuclear stress bodies rearranges and enhances NFIL3 to restrain acute inflammatory responses](https://doi.org/10.1016/j.cell.2025.05.003)
- [GSE277764](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE277764)のNanoporeのデータに加えて、上記の論文で登録されているデータ

- ストレス条件は以下の通り
- `For heat shock (HS) treatment, Human HeLa, THP-1 cells and HUVECs were replaced with pre-heated (42°C) conditioned medium and incubated in a culture incubator with an environmental temperature at 42°C and humidified 5% CO2 for 2 h, then recovered at 37°C for 1 h.`

- BioProject: [PRJNA1177810](https://www.ncbi.nlm.nih.gov/bioproject/PRJNA1177810)
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb) から移動
- PEP 形式のファイルには、run のアクセッション番号が記載されていない
- ChIP-Atlas には登録されていない

In [6]:
GSE280307_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE280307/GSE280307_PEP/GSE280307_PEP_raw.csv",
    separator=',',
    null_values=["null"]
).filter(
    (pl.col("treatment") == "HS stressed") |
    (pl.col("treatment") == "untreated")
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.col("big_key").alias("experiment_accession"),
    pl.lit("https://doi.org/10.1016/j.cell.2025.05.003").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(False).alias("is_chip_atlas_registered")
).rename(
    {
        "assembly": "genome_build"
    }
)

GSE280307 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE280307.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE280307_filtered = GSE280307[[s.name for s in GSE280307 if not (s.null_count() == GSE280307.height)]]

GSE280307_combined = GSE280307_PEP.join(
    GSE280307_filtered,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

GSE280307_combined = GSE280307_combined[[s.name for s in GSE280307_combined if not (s.null_count() == GSE280307_combined.height)]]

# display(GSE280307_combined)

In [7]:
defined_pairs_gse280307_atacseq = [
    # SRR31114891, SRR31114888
    ("SRX26497053", "SRX26497055"), # HS stressed vs. untreated 
    # SRR31114887, SRR31114886
    ("SRX26497056", "SRX26497058") # HS stressed vs. untreated
]

paired_metadata_df_gse280307_atacseq = create_paired_metadata(
    combined_df=GSE280307_combined,
    defined_sample_pairs=defined_pairs_gse280307_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse280307_atacseq is not None:
    paired_metadata_df_gse280307_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE280307_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse280307_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,cell_type,treatment,genome_build,supplementary_files_format_and_content,…,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX26497053""","""SRX26497055""","""atac-seq_thp-1_hs_rep1 vs atac-seq_thp-1_un_rep1""","""ATAC-seq_THP-1_HS_rep1 vs ATAC-seq_THP-1_UN_rep1""","""GSM8594499 vs GSM8594501""","""Public on May 26 2025""","""Oct 25 2024""","""May 26 2025""","""SRA""","""1""","""THP-1""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL24676""","""Li,,Yang""","""liyang_fudan@fudan.edu.cn""","""Institutes of Biological Sciences""","""Fudan University""","""131 Dong-An Road""","""Shanghai""","""200032""","""China""","""Illumina NovaSeq 6000""","""other""","""genomic""","""ATAC-seq""","""GSM8594499 vs GSM8594501""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8594nnn/GSM8594499/suppl/GSM8594499_ATAC-seq_THP-1_HS_rep1.CPM.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8594nnn/GSM8594501/suppl/GSM8594501_ATAC-seq_TH…","""GSE280307""","""0""","""SRX26497053 vs SRX26497055""","""THP-1""","""THP-1 cells""","""HS stressed vs untreated""","""CHM13 T2T v2.0""","""bigWig""",…,"""SRA1997279""","""other""","""2025-05-28""","""3392656984;3377789645 vs 3932490375;3859261283""","""ILLUMINA""","""missing""","""Generic""","""ATAC-seq_THP-1_HS_rep1 vs ATAC-seq_THP-1_UN_rep1""","""SRP540826""","""48355066 vs 55650730""","""ATAC-sequencing in untreated (UN), sodium arsenite (SA) and heat shock (HS) stressed THP-1 cells""","""GENERATED_FILE;GENERATED_FILE""","""ATAC-seq_THP-1_HS_rep1 vs ATAC-seq_THP-1_UN_rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8594499: ATAC-seq_THP-1_HS_rep1 Homo sapiens ATAC-seq vs Illumina NovaSeq 6000 sequencing: GSM8594501: ATAC-seq_THP-1_UN_rep1 Homo sapiens ATAC-seq""","""THP-1 cells""","""14506519800 vs 16695219000""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM8594499 vs GSM8594501""","""GSM8594499_r1 vs GSM8594501_r1""","""THP-1 cells were seeded at a density of 5×105 cells ml-1 in 6-well plates and exposing the cells to 200 ng ml-1 PMA) for 24 h. After SA, HS and control treatment, cells were centrifuged for 10 min at …","""missing""","""GSM8594499_r1 vs GSM8594501_r1""","""THP-1""","""GSM8594499 vs GSM8594501""","""env_tax:terrestrial;env_tax""","""2025-05-28""","""Homo sapiens""","""9606""","""PRJNA1177810""","""SRR31114891 vs SRR31114888""","""SRS23009971 vs SRS23009972""","""2025-05-28""","""GSE280307"""
"""SRX26497056""","""SRX26497058""","""atac-seq_thp-1_hs_rep2 vs atac-seq_thp-1_un_rep2""","""ATAC-seq_THP-1_HS_rep2 vs ATAC

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Strategy 1. GEO から 一旦リストを集める

- 全てのリストを調べるのは不可能なので、以下のクエリでヒットしたアクセッションを調べる

```text
((((((("heat shock response"[MeSH Terms] OR "heat shock"[Description]) OR "heat stress"[Description]) OR "high temperature stress"[Description]) OR "hyperthermia"[Description]) OR "heat stroke"[Description]) AND "homo sapiens"[Organism]) AND "genome binding/occupancy profiling by high throughput sequen"[Filter]) AND "gds sra"[Filter]
```

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. GSE101731 (ATAC-seq) (ChIP-Atlas: True)

- ✅ ChIP-Atlas: Diff Analysis
- ✅ ChIP-Atlas: Diff Analysis (via API)

&nbsp;

- ✅ DOWNLOAD (M5 Macbook Pro)
- ✅ trimming using `fastp` (M5 Macbook Pro)
- ✅ `.tagalign` using `Chromap` (M5 Macbook Pro)
- ✅ sort `.tagalign` for `ENCODE-rE2G` (M5 Macbook Pro)
<!-- - ✅ Execute `ENCODE-rE2G` (M5 Macbook Pro) -->

&nbsp;

- validate date: 2026-04-27
- [HSV-1-induced disruption of transcription termination resembles a cellular stress response but selectively increases chromatin accessibility downstream of genes](https://doi.org/10.1371/journal.ppat.1006954)
- ストレス条件は以下の通り
- `Heat stress was started by replacing the cell culture medium with pre-warmed 44°C medium and culturing the cells for 1 or 2h at 44°C.`
- mock の replicate が 4つあるので、これをコントロール群として使用する

In [8]:
GSE101731_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE101731/GSE101731_PEP/GSE101731_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "ATAC-seq") &
    (~pl.col("sample_title").str.contains("salt stress")) # 4 run data is salt stress
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.lit("https://doi.org/10.1371/journal.ppat.1006954").alias("reference_doi"),
    # branch temperature: mock samples are control, others are heat-stressed at 44 C
    pl.when(pl.col("sample_title").str.contains("(?i)mock"))
      .then(pl.lit("ctrl"))
      .otherwise(pl.lit("44 C"))
      .alias("stress treatment temperature"),
    # branch treatment time by sample_title (Mock / 1h / 2h)
    pl.when(pl.col("sample_title").str.contains("(?i)mock"))
      .then(pl.lit("ctrl"))
      .when(pl.col("sample_title").str.contains("1h"))
      .then(pl.lit("60 min."))
      .when(pl.col("sample_title").str.contains("2h"))
      .then(pl.lit("120 min."))
      .otherwise(pl.lit(None))
      .alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(True).alias("is_chip_atlas_registered")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)


GSE101731 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE101731.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE101731_filtered = GSE101731[[s.name for s in GSE101731 if not (s.null_count() == GSE101731.height)]]

GSE101731_combined = GSE101731_PEP.join(
    GSE101731_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

# pl.Config.set_tbl_rows(2)
# display(GSE101731_combined)

In [9]:
defined_pairs_gse101731_atacseq = [
    ("SRX3025783", "SRX3025779"), # 1h heat stress Rep 1 vs. Mock Rep 1
    ("SRX3025784", "SRX3025780"), # 1h heat stress Rep 2 vs. Mock Rep 2
    ("SRX3025783", "SRX3025781"), # 1h heat stress Rep 1 vs. Mock Rep 3
    ("SRX3025784", "SRX3025782"), # 1h heat stress Rep 2 vs. Mock Rep 4

    ("SRX3025785", "SRX3025779"), # 2h heat stress Rep 1 vs. Mock Rep 1
    ("SRX3025786", "SRX3025780"), # 2h heat stress Rep 2 vs. Mock Rep 2
    ("SRX3025785", "SRX3025781"), # 2h heat stress Rep 1 vs. Mock Rep 3
    ("SRX3025786", "SRX3025782") # 2h heat stress Rep 2 vs. Mock Rep 4
]

paired_metadata_df_gse101731_atacseq = create_paired_metadata(
    combined_df=GSE101731_combined,
    defined_sample_pairs=defined_pairs_gse101731_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse101731_atacseq is not None:
    # shape the column names
    paired_metadata_df_gse101731_atacseq = paired_metadata_df_gse101731_atacseq.with_columns(
        pl.col(["stress treatment temperature", "stress treatment time"])
        .str.strip_suffix(" vs ctrl") # https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.strip_suffix.html
    )

    paired_metadata_df_gse101731_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE101731_atacseq_paired_metadata.tsv",
        separator='\t'
    )
    pl.Config.set_tbl_rows(8)
    display(paired_metadata_df_gse101731_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_type,…,experiment_title,sample_accession_right,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX3025783""","""SRX3025779""","""1h_heat_stress_rep_1 vs mock_rep_1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR5856839 vs SRR5856835""","""1h heat stress Rep 1 vs Mock Rep 1""","""GSM2714164 vs GSM2714160""","""Public on Mar 08 2018""","""Jul 21 2017""","""May 15 2019""","""SRA""","""1""","""HFF_1h_44ºC_ATAC-seq vs HFF_mock_ATAC-seq""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL18573""","""Caroline,,Friedel""","""Institut für Informatik""","""Ludwig-Maximilians-Universität München""","""Amalienstr. 17""","""München""","""80333""","""Germany""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM2714164 vs GSM2714160""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2714nnn/GSM2714164/suppl/GSM2714164_heat_1h_1.bed.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2714nnn/GSM2714160/suppl/GSM2714160_Mock_1.bed.gz""","""GSE101731""","""0""","""SRX3025783 vs SRX3025779""","""Primary human foreskin fibroblasts (HFF)""",…,"""NextSeq 500 sequencing: GSM2714164: 1h heat stress Rep 1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM2714160: Mock Rep 1 Homo sapiens ATAC-seq""","""SAMN07375245 vs SAMN07375249""","""Homo sapiens""","""SRA590045""","""other""","""2018-03-10""","""1729702908;1745134169 vs 1226000309;1248603430""","""ILLUMINA""","""Generic""","""1h heat stress Rep 1 vs Mock Rep 1""","""SRP113284""","""42335982 vs 29415519""","""ATAC-seq of HFF exposed to salt and heat stress""","""GENERATED_FILE;GENERATED_FILE""","""1h heat stress Rep 1 vs Mock Rep 1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM2714164: 1h heat stress Rep 1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM2714160: Mock Rep 1 Homo sapiens ATAC-seq""","""Primary human foreskin fibroblasts (HFF)""","""6435069264 vs 4471158888""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM2714164 vs GSM2714160""","""GSM2714164_r1 vs GSM2714160_r1""","""Samples were processed according to the original ATAC-seq protocol (Buenrostro et al. Nature Methods 2013) See Buenrostro et al, Nature Methods 2013 NextSeq 500 sequencing 75bp paired-end reads""","""GSM2714164 vs GSM2714160""","""GEO""","""env_tax:terrestrial;env_tax""","""2018-03-10""","""Homo sapiens""","""9606""","""PRJNA395312""","""SRR5856839 vs SRR5856835""","""SRS2374866 vs SRS2374862""","""2018-03-10""","""GSE101731"""
"""SRX3025784""","""SRX3025780""","""1h_heat_stress_rep_2 vs mock_rep_2""","""other""","""Homo s

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE154744 (ATAC-seq) (ChIP-Atlas: True)

- ✅ ChIP-Atlas: Diff Analysis (only no pre-conditioning)

&nbsp;

- ✅ DOWNLOAD (M5 Macbook Pro)
- ✅ trimming using `fastp` (M5 Macbook Pro)
- ✅ `.tagalign` using `Chromap` (M5 Macbook Pro)
- ✅ sort `.tagalign` for `ENCODE-rE2G` (M5 Macbook Pro)
<!-- - ✅ Execute `ENCODE-rE2G` (M5 Macbook Pro) -->

&nbsp;

- validate date: 2026-04-28
- [Stress-induced transcriptional memory accelerates promoter-proximal pause release and decelerates termination over mitotic divisions](https://doi.org/10.1016/j.molcel.2021.03.007)
- contorol に該当するATAC-seq のデータのなかに、pre-conditioning (`pc`) という既に 熱ストレスを加えている状態のデータがあるため、ChIP-Atlas の `diff analysis`をやるなら、これらのデータは今回の解析には使わないことに決定
- つまり、`uc0` をコントロールのデータとして使用することとする

In [10]:
GSE154744_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE154744/GSE154744_PEP/GSE154744_PEP_raw.csv",
    separator=','
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.lit("https://doi.org/10.1016/j.molcel.2021.03.007").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    # branch additional heat-shock duration by sample_name (uC0/pC0 / 12p5 / 25 / 30 / 40 / 60)
    pl.when(pl.col("sample_name").str.contains("(?i)_uc0_|_pc0_"))
      .then(pl.lit("ctrl"))
      .when(pl.col("sample_name").str.contains("12p5"))
      .then(pl.lit("12.5 min."))
      .when(pl.col("sample_name").str.contains("(?i)_uc25|_pc25"))
      .then(pl.lit("25 min."))
      .when(pl.col("sample_name").str.contains("(?i)_uc30|_pc30"))
      .then(pl.lit("30 min."))
      .when(pl.col("sample_name").str.contains("(?i)_uc40|_pc40"))
      .then(pl.lit("40 min."))
      .when(pl.col("sample_name").str.contains("(?i)_uc60|_pc60"))
      .then(pl.lit("60 min."))
      .otherwise(pl.lit(None))
      .alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(True).alias("is_chip_atlas_registered")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE154744 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE154744.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE154744_filtered = GSE154744[[s.name for s in GSE154744 if not (s.null_count() == GSE154744.height)]]

GSE154744_combined = GSE154744_PEP.join(
    GSE154744_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)

# pl.Config.set_tbl_cols(100)
# display(GSE154744_combined)

In [11]:
defined_pairs_gse154744_atacseq = [
    # pre-conditioning: none
    # ChIP-Atlas: Diff Analysis: Complete
    ("SRX8772746", "SRX8772744"), # atacseq_k562_s_uc12p5_rep1 vs. atacseq_k562_s_uc0_rep1
    ("SRX8772747", "SRX8772745"), # atacseq_k562_s_uc12p5_rep2 vs. atacseq_k562_s_uc0_rep2

    ("SRX8772748", "SRX8772744"), # atacseq_k562_s_uc25_rep1 vs. atacseq_k562_s_uc0_rep1
    ("SRX8772749", "SRX8772745"), # atacseq_k562_s_uc25_rep2 vs. atacseq_k562_s_uc0_rep2

    ("SRX8772750", "SRX8772744"), # atacseq_k562_s_uc40_rep1 vs. atacseq_k562_s_uc0_rep1
    ("SRX8772751", "SRX8772745"), # atacseq_k562_s_uc40_rep2 vs. atacseq_k562_s_uc0_rep2


    # pre-conditioning: 1h heat shock at 42C water bath, followed by 48 h recovery at 37 C
    # ("SRX8772754", "SRX8772752"), # atacseq_k562_s_pc12p5_rep1 vs. atacseq_k562_s_pc0_rep1
    # ("SRX8772755", "SRX8772753"), # atacseq_k562_s_pc12p5_rep2 vs. atacseq_k562_s_pc0_rep2
    # ("SRX8772756", "SRX8772752"), # atacseq_k562_s_pc25_rep1 vs. atacseq_k562_s_pc0_rep1
    # ("SRX8772757", "SRX8772753"), # atacseq_k562_s_pc25_rep2 vs. atacseq_k562_s_pc0_rep2
    # ("SRX8772732", "SRX8772752"), # atacseq_k562_s_pc40_rep1 vs. atacseq_k562_s_pc0_rep1
    # ("SRX8772733", "SRX8772753"), # atacseq_k562_s_pc40_rep2 vs. atacseq_k562_s_pc0_rep2

    # pre-conditioning: none
    # ChIP-Atlas: Diff Analysis: Complete
    ("SRX8772736", "SRX8772734"), # atacseq_k562_hss_uc30_rep1 vs. atacseq_k562_hss_uc0_rep1
    ("SRX8772737", "SRX8772735"), # atacseq_k562_hss_uc30_rep2 vs. atacseq_k562_hss_uc0_rep2
    
    ("SRX8772738", "SRX8772734"), # atacseq_k562_hss_uc60_rep1 vs. atacseq_k562_hss_uc0_rep1
    ("SRX8772739", "SRX8772735"), # atacseq_k562_hss_uc60_rep2 vs. atacseq_k562_hss_uc0_rep2

    # pre-conditioning: 9 x 1h heat shock at 42C (divided to 3 consecutive days), followed by 48 h recovery at 37 C
    # ("SRX8772742", "SRX8772740"), # atacseq_k562_hss_pc30_rep1 vs. atacseq_k562_hss_pc0_rep1
    # ("SRX8772743", "SRX8772741"), # atacseq_k562_hss_pc30_rep2 vs. atacseq_k562_hss_pc0_rep2
    # ("SRX8772730", "SRX8772740"), # atacseq_k562_hss_pc60_rep1 vs. atacseq_k562_hss_pc0_rep1
    # ("SRX8772731", "SRX8772741"), # atacseq_k562_hss_pc60_rep2 vs. atacseq_k562_hss_pc0_rep2
]

paired_metadata_df_gse154744_atacseq = create_paired_metadata(
    combined_df=GSE154744_combined,
    defined_sample_pairs=defined_pairs_gse154744_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse154744_atacseq is not None:

    # shape the column names
    paired_metadata_df_gse154744_atacseq = paired_metadata_df_gse154744_atacseq.with_columns(
    pl.col(["stress treatment temperature", "stress treatment time"])
    .str.strip_suffix(" vs ctrl") # https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.str.strip_suffix.html
    )
    paired_metadata_df_gse154744_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE154744_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    pl.Config.set_tbl_rows(2)
    display(paired_metadata_df_gse154744_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_characteristics_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,…,experiment_title,sample_accession_right,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX8772746""","""SRX8772744""","""atacseq_k562_s_uc12p5_rep1 vs atacseq_k562_s_uc0_rep1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR12267673 vs SRR12267671""","""ATACseq_K562_s_uC12p5_rep1 vs ATACseq_K562_s_uC0_rep1""","""GSM4678111 vs GSM4678109""","""Public on Apr 01 2021""","""Jul 20 2020""","""Apr 01 2021""","""SRA""","""1""","""K562""","""Homo sapiens""","""9606""","""heat shock (additional to preconditioning): 12.5 min at 42C water bath (initiated with prewarmed preconditioned media) vs heat shock (additional to preconditioning): no heat stress applied""","""genomic DNA""","""Normalized_ATACseq_K562_s_uC12p5_BRs.bigWig vs Normalized_ATACseq_K562_s_uC0_BRs.bigWig""","""GPL18573""","""Anniina,,Vihervaara""","""mavih@kth.se""","""+358 50 5463694""","""Vihervaara""","""Gene Technology""","""KTH Royal Institute of Technology""","""Tomtebodavägen 23A""","""Stockholm""","""17165""","""Sweden""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM4678111 vs GSM4678109""",…,"""NextSeq 500 sequencing: GSM4678111: ATACseq_K562_s_uC12p5_rep1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM4678109: ATACseq_K562_s_uC0_rep1 Homo sapiens ATAC-seq""","""SAMN15582298 vs SAMN15582241""","""Chromatin accessibility in human K562 erythroleukemia cells profiled upon heat shock, recovery, and heat-induced transcriptional memory""","""SRA1100912""","""other""","""2021-04-02""","""381746585;382174700 vs 121681352;134798277""","""ILLUMINA""","""Generic""","""ATACseq_K562_s_uC12p5_rep1 vs ATACseq_K562_s_uC0_rep1""","""SRP272606""","""16162574 vs 4929411""","""Chromatin accessibility in human K562 erythroleukemia cells profiled upon heat shock, recovery, and heat-induced transcriptional memory""","""GENERATED_FILE;GENERATED_FILE""","""ATACseq_K562_s_uC12p5_rep1 vs ATACseq_K562_s_uC0_rep1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM4678111: ATACseq_K562_s_uC12p5_rep1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM4678109: ATACseq_K562_s_uC0_rep1 Homo sapiens ATAC-seq""","""1228355624 vs 374635236""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM4678111 vs GSM4678109""","""GSM4678111_r1 vs GSM4678109_r1""","""After treatments, the cells were harvested by centrifugation, and washed twice with PBS. Chromatin was isolated as in Chu et al., 2018, Nature Genetics, 50: 1553–1564 and run-on reactions performed wi…","""GSM467

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. GSE209675 (ATAC-seq) ChIP-atlas: True

- ✅ ChIP-Atlas: Diff Analysis

&nbsp;

- ✅ DOWNLOAD (M5 Macbook Pro)
<!-- - ✅ trimming using `fastp` (M5 Macbook Pro) -->
<!-- - ✅ `.tagalign` using `Chromap` (M5 Macbook Pro) -->
<!-- - ✅ sort `.tagalign` for `ENCODE-rE2G` (M5 Macbook Pro) -->
<!-- - ✅ Execute `ENCODE-rE2G` (M5 Macbook Pro) -->

&nbsp;

- validate date: 2026-04-28
- [Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding.](https://doi.org/10.1038/s41467-023-43157-7)
- 熱ストレス条件は以下の通り
- `Heat-Shock treatment was started by placing a dish onto a heated water surface for 1 min to raise the temperature quickly and continued in a 42 °C 5% CO2 incubator for the remaining time.`
- `A 60-min incubation at 42 °C induced a control HSP gene seen as spreading of the Pol II and nascent RNA signal into the gene body (Fig. 1a).`

In [12]:
GSE209675_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE209675/GSE209675_PEP/GSE209675_PEP_raw.csv",
    separator=','
).filter(
    pl.col("treatment") != "Arsenic"
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.col("big_key").alias("experiment_accession"),
    pl.lit("https://doi.org/10.1038/s41467-023-43157-7").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(True).alias("is_chip_atlas_registered")
).rename(
    {
        "assembly": "genome_build"
    }
)

GSE209675 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE209675.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE209675_filtered = GSE209675[[s.name for s in GSE209675 if not (s.null_count() == GSE209675.height)]]

GSE209675_combined = GSE209675_PEP.join(
    GSE209675_filtered,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

# display(GSE209675_PEP)

In [13]:
defined_pairs_gse209675_atacseq = [
    ("SRX16651805", "SRX16651804"), # mcf7_atac_hs_rep1 vs. mcf7_atac_nhs_rep1
    ("SRX16651808", "SRX16651807")  # mcf7_atac_hs_rep2 vs. mcf7_atac_nhs_rep2
]

paired_metadata_df_gse209675_atacseq = create_paired_metadata(
    combined_df=GSE209675_combined,
    defined_sample_pairs=defined_pairs_gse209675_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse209675_atacseq is not None:
    paired_metadata_df_gse209675_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE209675_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse209675_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,…,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX16651805""","""SRX16651804""","""mcf7_atac_hs_rep1 vs mcf7_atac_nhs_rep1""","""null""","""null""","""null""","""null""","""null""","""null""","""MCF7 ATAC HS rep1 vs MCF7 ATAC NHS rep1""","""GSM6383319 vs GSM6383318""","""Public on Oct 30 2023""","""Jul 25 2022""","""Oct 30 2023""","""SRA""","""1""","""MCF7""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL20301""","""Sergei,,Nechaev""","""sergei.nechaev@und.edu""","""Nechaev Lab""","""Biomedical Sciences""","""University of North Dakota""","""Grand Forks""","""North Dakota""","""58202""","""USA""","""Illumina HiSeq 4000""","""other""","""genomic""","""ATAC-seq""","""GSM6383319 vs GSM6383318""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383319/suppl/GSM6383319_MCF7_ATAC_HS_ATACseq_rep1.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383318/suppl/GSM6383318_MCF7_ATAC_NH…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383319/suppl/GSM6383319_MCF7_ATAC_HS_ATACseq_rep1.narrowPeak.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6383nnn/GSM6383318/suppl/GSM6383318_M…",…,"""SRA1462025""","""other""","""2023-11-01""","""2071701475;2063678616 vs 1831808926;1830205036""","""ILLUMINA""","""missing""","""Generic""","""MCF7 ATAC HS rep1 vs MCF7 ATAC NHS rep1""","""SRP387967""","""42798658 vs 36508983""","""Transcriptional responses of cancer cells to heat shock-inducing stimuli involve amplification of robust HSF1 binding""","""GENERATED_FILE;GENERATED_FILE""","""MCF7 ATAC HS rep1 vs MCF7 ATAC NHS rep1""","""Illumina HiSeq 4000""","""Illumina HiSeq 4000 sequencing: GSM6383319: MCF7 ATAC HS rep1 Homo sapiens ATAC-seq vs Illumina HiSeq 4000 sequencing: GSM6383318: MCF7 ATAC NHS rep1 Homo sapiens ATAC-seq""","""mammary gland, adenocarcinoma""","""12839597400 vs 10952694900""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM6383319 vs GSM6383318""","""GSM6383319_r1 vs GSM6383318_r1""","""Fresh cells were subjected to ATAC library construction protocol ATAC-seq was done using the original protocol and buffer {Buenrostro et al., 2013, #304248}. Incubation with Tn5 transposase was done i…","""missing""","""GSM6383319_r1 vs GSM6383318_r1""","""MCF7""","""GSM6383319 vs GSM6383318""","""env_tax:terrestrial;env_tax""","""2023-11-01""","""Homo sapiens""","""9606""","""PRJNA862089""","""SRR20628057 vs SRR20628058""","""SRS14286140 vs SRS14286139""","""2023-11-01""","""GSE209675"""
"""SRX16651808""","""SRX16651807""",

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. GSE267931 (ATAC-seq) ChIP-Atlas: True

- ✅ ChIP-Atlas: Diff Analysis (Error)

&nbsp;

- [Inflammatory stress-mediated chromatin changes underlie dysfunction in endothelial cells (BioRxiv)](https://doi.org/10.1101/2023.10.11.561959)
- `Diff Analysis` はエラーのため、実行できなかった
- [SRX24613861](https://chip-atlas.org/view?id=SRX24613861) のゲノムのバージョンが `hg19`のみが ChIP-Atlasに登録されていたため、実行エラーになっている

In [14]:
GSE267931_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE267931/GSE267931_PEP/GSE267931_PEP_raw.csv",
    separator=','
).filter(
    pl.col("treatment") != "TNF-alpha"
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.col("big_key").alias("experiment_accession"),
    pl.lit("https://doi.org/10.1101/2023.10.11.561959").alias("reference_doi"),
    pl.lit("39 C").alias("stress treatment temperature"),
    pl.lit("120 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(True).alias("is_chip_atlas_registered")
).rename(
    {
        "assembly": "genome_build"
    }
)

GSE267931 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE267931.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE267931_filtered = GSE267931[[s.name for s in GSE267931 if not (s.null_count() == GSE267931.height)]]

GSE267931_combined = GSE267931_PEP.join(
    GSE267931_filtered,
    on=["experiment_accession"],
    how="left",
    coalesce=True
)

# pl.Config.set_tbl_cols(100)
# display(GSE267931_combined)

In [15]:
defined_pairs_gse267931_atacseq = [
    ("SRX24613860", "SRX24613858"), # huvec_atac-seq_39_°c_heat_stress_for_2_hours_biological_replicate_1_hs1_ vs. huvec_atac-seq_control_biological_replicate_1_c1_
    ("SRX24613861", "SRX24613859")  # huvec_atac-seq_39_°c_heat_stress_for_2_hours_biological_replicate_2_hs2_ vs. huvec_atac-seq_control_biological_replicate_2_c2_
]

paired_metadata_df_gse267931_atacseq = create_paired_metadata(
    combined_df=GSE267931_combined,
    defined_sample_pairs=defined_pairs_gse267931_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse267931_atacseq is not None:
    paired_metadata_df_gse267931_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE267931_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse267931_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,…,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX24613860""","""SRX24613858""","""huvec_atac-seq_39_°c_heat_stress_for_2_hours_biological_replicate_1_hs1_ vs huvec_atac-seq_control_biological_replicate_1_c1_""","""null""","""null""","""null""","""null""","""null""","""null""","""HUVEC, ATAC-Seq, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, ATAC-Seq, control, biological replicate 1 (C1)""","""GSM8281745 vs GSM8281743""","""Public on May 25 2024""","""May 20 2024""","""May 25 2024""","""SRA""","""1""","""HUVEC""","""Homo sapiens""","""9606""","""genomic DNA""","""GPL24676""","""Haibo,,Liu""","""haibo.liu@umassmed.edu""","""5153573719""","""MCCB""","""UMass Chan""","""164 Plantation St""","""Unionville""","""CT""","""6085""","""USA""","""Illumina NovaSeq 6000""","""other""","""genomic""","""ATAC-seq""","""GSM8281745 vs GSM8281743""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8281nnn/GSM8281745/suppl/GSM8281745_HS1.bigwig vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8281nnn/GSM8281743/suppl/GSM8281743_C1.bigwig""",…,"""SRA1871865""","""other""","""2024-06-30""","""1977291880;2044419792 vs 1655006674;1626851023""","""ILLUMINA""","""missing""","""Generic""","""HUVEC, ATAC-Seq, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, ATAC-Seq, control, biological replicate 1 (C1)""","""SRP508795""","""64418675 vs 51047533""","""Inflammatory stress-mediated chromatin changes underlie dysfunction in endothelial cells. [ATAC-Seq]""","""GENERATED_FILE;GENERATED_FILE""","""HUVEC, ATAC-Seq, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, ATAC-Seq, control, biological replicate 1 (C1)""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8281745: HUVEC, ATAC-Seq, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) Homo sapiens ATAC-seq vs Illumina NovaSeq 6000 sequencing: GSM8281743: HUVEC,…","""vascular endothelial""","""6570704850 vs 5206848366""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM8281745 vs GSM8281743""","""GSM8281745_r1 vs GSM8281743_r1""","""HUVEC cells were grown in dishes, cryopreserved in 1ml of media with DMSO containing 0.5-1.5x10^6 cells. The experiment was done in biological duplicate, paired with RNA-seq experiment. The ATAC-seq e…","""missing""","""GSM8281745_r1 vs GSM8281743_r1""","""HUVEC""","""GSM8281745 vs GSM8281743""","""env_tax:terrestrial;env_tax""","""2024-06-30""","""Homo sapiens""","""9606""","""PRJNA1113657""","""SRR29089716 vs SRR

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Strategy 2: `GEOfinder` で検索漏れがないか調査する

- [`GEOfinder`](https://bioapps.byu.edu/geofinder/) で、シンプルに見逃しているものがないか調査する
- 出力結果をPDFとして保存し、ChatGPT (GPT-5.5) でざっくり探してもらった

&nbsp;

## 7. GSE173536 (ATAC-seq) ChIP-Atlas: True

- ✅ ChIP-Atlas: Diff Analysis

&nbsp;

- validate date: 2027-04-28
- [Characterizing primary transcriptional responses to short term heat shock in Down syndrome](https://doi.org/10.1371/journal.pone.0307375)

In [16]:
GSE173536_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE173536/GSE173536_PEP/GSE173536_PEP_raw.csv",
    separator=','
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.lit("https://doi.org/10.1371/journal.pone.0307375").alias("reference_doi"),
    pl.lit("42 C").alias("stress treatment temperature"),
    pl.lit("60 min.").alias("stress treatment time"),
    pl.lit("37 C (inferred)").alias("control treatment temperature"),
    pl.lit(True).alias("is_chip_atlas_registered")
).rename(
    {
        "srr": "run_accession",
        "srx": "experiment_accession"
    }
)

GSE173536 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE173536.metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE173536_filtered = GSE173536[[s.name for s in GSE173536 if not (s.null_count() == GSE173536.height)]]

GSE173536_combined = GSE173536_PEP.join(
    GSE173536_filtered,
    on=["run_accession", "experiment_accession"],
    how="left",
    coalesce=True
)


# pl.Config.set_tbl_rows(14)
# pl.Config.set_tbl_cols(100)
# display(GSE173536_combined)

In [17]:
defined_pairs_gse173536_atacseq = [
    # Disomic cells (D21)
    ("SRX10705939", "SRX10705938"), # ATACseq_Lymphblastoid cells_D21_42C_rep1 vs. ATACseq_Lymphblastoid cells_D21_37C_rep1
    ("SRX10705946", "SRX10705944"), # ATACseq_Lymphblastoid cells_D21_42C_rep2_technical1 vs. ATACseq_Lymphblastoid cells_D21_37C_rep2_technical1
    ("SRX10705947", "SRX10705945"), # ATACseq_Lymphblastoid cells_D21_42C_rep2_technical2 vs. ATACseq_Lymphblastoid cells_D21_37C_rep2_technical2

    # Trisomic cells (T21)
    ("SRX10705941", "SRX10705940"), # ATACseq_Lymphblastoid cells_T21_42C_rep1 vs. ATACseq_Lymphblastoid cells_T21_37C_rep1
    ("SRX10705950", "SRX10705948"), # ATACseq_Lymphblastoid cells_T21_42C_rep2_technical1 vs. ATACseq_Lymphblastoid cells_T21_37C_rep2_technical1
    ("SRX10705951", "SRX10705949")  # ATACseq_Lymphblastoid cells_T21_42C_rep2_technical2 vs. ATACseq_Lymphblastoid cells_T21_37C_rep2_technical2
]

paired_metadata_df_gse173536_atacseq = create_paired_metadata(
    combined_df=GSE173536_combined,
    defined_sample_pairs=defined_pairs_gse173536_atacseq,
    sample_id_column_name="experiment_accession"
)

if paired_metadata_df_gse173536_atacseq is not None:
    paired_metadata_df_gse173536_atacseq.write_csv(
        "../../Data/metadata/metadata_atacseq/HS_related_paired_metadata/GSE173536_atacseq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse173536_atacseq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,run_accession,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,…,experiment_title,sample_accession_right,project_name,submission_accession,library_selection,last_updated,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,fastq_file_role,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,tag,first_created,scientific_name,tax_id,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRX10705939""","""SRX10705938""","""atacseq_lymphblastoid_cells_d21_42c_rep1 vs atacseq_lymphblastoid_cells_d21_37c_rep1""","""other""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR14352092 vs SRR14352091""","""ATACseq_Lymphblastoid cells_D21_42C_rep1 vs ATACseq_Lymphblastoid cells_D21_37C_rep1""","""GSM5269424 vs GSM5269423""","""Public on Apr 21 2023""","""Apr 28 2021""","""Apr 21 2023""","""SRA""","""1""","""lymphoblastoid-EBV immortalized""","""Homo sapiens""","""9606""","""genomic DNA""","""Eric_D21, genomic DNA (ATAC-seq)""","""GPL18573""","""Mary,Ann,Allen""","""mary.a.allen@colorado.edu""","""608-209-6921""","""Allen""","""BioFrontiers""","""University of Colorado""","""UCB 596 JSCBB""","""Boulder""","""Co""","""80309""","""USA""","""Illumina NextSeq 500""","""other""","""genomic""","""ATAC-seq""","""GSM5269424 vs GSM5269423""",…,"""NextSeq 500 sequencing: GSM5269424: ATACseq_Lymphblastoid cells_D21_42C_rep1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM5269423: ATACseq_Lymphblastoid cells_D21_37C_rep1 Homo sapiens ATAC-seq""","""SAMN18915529 vs SAMN18915530""","""Blood cells with trisomy 21 overreact to heat shock stress [ATAC-seq]""","""SRA1225196""","""other""","""2023-05-06""","""386160758;386523331 vs 480559686;481149630""","""ILLUMINA""","""Generic""","""ATACseq_Lymphblastoid cells_D21_42C_rep1 vs ATACseq_Lymphblastoid cells_D21_37C_rep1""","""SRP316796""","""16830219 vs 21012017""","""Blood cells with trisomy 21 overreact to heat shock stress [ATAC-seq]""","""GENERATED_FILE;GENERATED_FILE""","""ATACseq_Lymphblastoid cells_D21_42C_rep1 vs ATACseq_Lymphblastoid cells_D21_37C_rep1""","""NextSeq 500""","""NextSeq 500 sequencing: GSM5269424: ATACseq_Lymphblastoid cells_D21_42C_rep1 Homo sapiens ATAC-seq vs NextSeq 500 sequencing: GSM5269423: ATACseq_Lymphblastoid cells_D21_37C_rep1 Homo sapiens ATAC-seq""","""lymphoblastoid""","""1245436206 vs 1554889258""","""ATAC-seq""","""GENOMIC""","""PAIRED""","""public""","""GSM5269424 vs GSM5269423""","""GSM5269424_r1 vs GSM5269423_r1""","""Suspension cells were then used to prepare an ATAC-seq library using the OMNI-ATAC-seq protocol. For control samples in calling ATAC-seq peaks, genomic DNA was extracted from each of the two cell line…","""GSM5269424 vs GSM5269423""","""GEO""","""env_tax:terrestrial;env_tax""","""2023-05-05""","""Homo sapiens""","""9606""","""PRJNA725915""","""SRR14352092 vs SRR14352091""","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Strategy 3: GEOにないが、BioProject のエントリーがあるものを調査する

- `TogoID`で `bioproject` -> `geo_series_id` で対応がつかなかったものを調査
- 今回の解析では、なし と判断

In [18]:
# chipatlas_atacseq_bioproject_no_geo_list = pl.read_csv(
#     "../../out/fantabio/ChIP-atlas_enrichment_result_lncRNA_ela_202604/ATAC-seq_enrichment/up5p_lncRNA_ela_atacseq_enrichment_list_bioproject_geo.tsv",
#     separator='\t'
# ).filter(
#     (pl.col("geo_series_id").is_null()) & (pl.col("bioproject_id").is_not_null())
# ).drop(
#     "geo_series_id"
# ).rename(
#     {
#         "sra_experiment_id": "experiment_accession",
#         "bioproject_id": "study_accession"
#     }
# )

# display(chipatlas_atacseq_bioproject_no_geo_list)

In [19]:
# metadata_atacseq_dir = pathlib.Path("../../Data/metadata/metadata_atacseq")
# metadata_atacseq_files = sorted(metadata_atacseq_dir.glob("*.metadata.tsv"))

# # Read every per-project metadata TSV with all columns as Utf8.
# # This avoids dtype mismatches across files when the same column is
# # inferred as different types in different projects.
# atacseq_metadata_all = pl.concat(
#     [
#         pl.read_csv(
#             f,
#             separator='\t',
#             ignore_errors=True,
#             quote_char=None,
#             infer_schema_length=0,
#         )
#         for f in metadata_atacseq_files
#     ],
#     how="diagonal_relaxed",
# )

# pl.Config.set_tbl_rows(2)
# print(f"loaded {len(metadata_atacseq_files)} files / total rows: {atacseq_metadata_all.height} / total cols: {atacseq_metadata_all.width}")
# display(atacseq_metadata_all)

In [20]:
# # Left-join the chipatlas list (key pairs) against the concatenated metadata
# # on (experiment_accession, study_accession) to attach per-experiment metadata.
# chipatlas_atacseq_bioproject_metadata = chipatlas_atacseq_bioproject_no_geo_list.join(
#     atacseq_metadata_all,
#     on=["experiment_accession", "study_accession"],
#     how="inner",
# )

# chipatlas_atacseq_bioproject_metadata_filtered = chipatlas_atacseq_bioproject_metadata[[s.name for s in chipatlas_atacseq_bioproject_metadata if not (s.null_count() == chipatlas_atacseq_bioproject_metadata.height)]]

# # quick sanity check: how many key pairs got at least one match
# matched_pairs = (
#     chipatlas_atacseq_bioproject_metadata
#     .filter(pl.col("run_accession").is_not_null())
#     .select(["experiment_accession", "study_accession"])
#     .unique()
#     .height
# )
# total_pairs = chipatlas_atacseq_bioproject_no_geo_list.height
# print(f"matched key pairs: {matched_pairs} / {total_pairs}")

# chipatlas_atacseq_bioproject_metadata_filtered.write_csv(
#     "../../out/fantabio/ChIP-atlas_enrichment_result_lncRNA_ela_202604/ATAC-seq_enrichment/chipatlas_atacseq_bioproject_no_geo_metadata_filtered.tsv",
#     separator='\t'
# )

# display(chipatlas_atacseq_bioproject_metadata_filtered)

In [21]:
# # 熱ストレスに関連していそうなデータをフィルタリング
# # なさそう
# chipatlas_atacseq_bioproject_metadata_filtered_hs = chipatlas_atacseq_bioproject_metadata_filtered.filter(
#     (pl.col("sample_title").str.contains("heat shock")) |
#     (pl.col("sample_title").str.contains("heat stress")) |
#     (pl.col("sample_title").str.contains("HS")) |
#     (pl.col("sample_title").str.contains("heat")) |
#     (pl.col("sample_title").str.contains("hyperthermia")) |
#     (pl.col("sample_description").str.contains("heat shock")) |
#     (pl.col("sample_description").str.contains("heat stress")) |
#     (pl.col("sample_description").str.contains("HS")) |
#     (pl.col("sample_description").str.contains("heat")) |
#     (pl.col("sample_description").str.contains("hyperthermia"))
# )

# pl.Config.set_tbl_rows(15)
# display(chipatlas_atacseq_bioproject_metadata_filtered_hs)

&nbsp;

&nbsp;

## Concat Dataframe

In [22]:
dfs_atacseq_all: list[pl.DataFrame | None] = [
    paired_metadata_df_gse105028_atacseq,
    paired_metadata_df_gse280307_atacseq,
    paired_metadata_df_gse101731_atacseq,
    paired_metadata_df_gse154744_atacseq,
    paired_metadata_df_gse209675_atacseq,
    paired_metadata_df_gse267931_atacseq,
    paired_metadata_df_gse173536_atacseq,
]


dfs_atacseq:  list[pl.DataFrame] = [df for df in dfs_atacseq_all if df is not None]

In [23]:
drop_columns_list_v2 = [
    "sample_channel_count",
    "sample_contact_name",
    "sample_contact_email",
    "sample_contact_phone",
    "sample_contact_institute",
    "sample_contact_address",
    "sample_contact_city",
    "sample_contact_state",
    "sample_contact_zip_postal_code",
    "sample_contact_country",
    "sample_data_row_count",
    "sample_contact_department",
    "supplementary_files_format_and_content",
    "sample_contact_laboratory",
    "sample_contact_name",
    "biosample",
    "sra",
    "srr",
    "srx",
    "sample_supplementary_file_2",
    "sample_supplementary_file_1",
    "big_key",
    "experiment_title",
    "sample_accession_right",
    "project_name",
    "submission_accession",
    "last_updated",
    "fastq_bytes",
    "instrument_platform",
    "ncbi_reporting_standard",
    "sample_description_right",
    "read_count",
    "fastq_file_role",
    "sample_title_right",
    "instrument_model",
    "base_count",
    "status",
    "sample_alias",
    "run_alias",
    "tag",
    "first_created",
    "accession",
    "first_public",
    "study_alias",
    "country",
    "cell_type_right",
    "collection_date",
    "cell_line_right",
    "library_name"
]

In [24]:
atacseq_concat = pl.concat(dfs_atacseq, how="diagonal_relaxed").drop(
    drop_columns_list_v2
)

atacseq_concat.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/atacseq_metadata_all_{execute_date}.tsv",
    separator='\t'
)